# One Tool, Three Uses: AutoGluon MultiModalPredictor

**MIDAS AI in Research Handbook — Multimodal Learning with AutoGluon**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/autogluon_multimodal_demo.ipynb)

This notebook demonstrates AutoGluon's `MultiModalPredictor` across three different use cases: text only, images only, and text combined with structured tabular features. The core insight is that the calling interface stays the same regardless of what you feed it — AutoGluon detects the data types and routes each column to the appropriate pretrained model automatically.

The three sections build on each other:

| Section | Data type | Dataset | Task |
|---|---|---|---|
| Part 1 | Text only | Stanford SST-2 | Sentiment classification |
| Part 2 | Images only | Beans (plant disease) | Disease classification |
| Part 3 | Text + tabular | Wine Reviews | Quality score prediction |

If you have not already worked through the tabular tutorial in Chapter 17, it is worth doing that first. This notebook assumes you are familiar with the split-fit-evaluate pattern and focuses on what changes when you move beyond structured columns.


## Step 1 — Enable GPU

**Part 2 (image classification) requires a GPU.** Parts 1 and 3 will work on CPU, though they will be slower.

Before running anything else: **Runtime > Change runtime type > T4 GPU**, then run the cell below to confirm.


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print("You are all set for all three sections.")
else:
    print("No GPU detected.")
    print("Parts 1 and 3 will still run, but Part 2 (image classification) will be very slow.")
    print("To enable GPU: Runtime > Change runtime type > T4 GPU")

## Step 2 — Install AutoGluon and Dependencies

This installs the multimodal module along with the datasets library for loading data from Hugging Face. Run once per Colab session.


In [ ]:
!pip install -q autogluon.multimodal datasets Pillow 2>&1 | tail -3
print("Installation complete.")

## ⚠️ Restart Your Runtime Before Continuing

**After the install cell finishes, you must restart the runtime before running any imports.**

Why: Colab keeps previously loaded versions of PyTorch and torchvision in memory. Installing AutoGluon pulls in updated versions, and if the old modules are still cached, you will get a `circular import` error when you try to import `MultiModalPredictor`.

In Colab: **Runtime > Restart session** (or Ctrl+M then .)

After restarting, skip the install cell above (it is already done) and continue from the GPU check below.

## Step 4 — Imports


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from autogluon.multimodal import MultiModalPredictor
from datasets import load_dataset

print("Ready.")

---

## Part 1: Text Only — Sentiment Classification

The dataset here is Stanford Sentiment Treebank (SST-2), a standard benchmark for sentence-level sentiment classification (Socher et al., 2013). Each row is a short movie review sentence labeled as positive (1) or negative (0).

We use a 600-row subset to keep training time short. The goal is not to beat published benchmarks — it is to show how `MultiModalPredictor` handles a DataFrame that contains only a text column and a label.

In [ ]:
# Load SST-2 from Hugging Face
sst2 = load_dataset("stanfordnlp/sst2", split="train").to_pandas()

# Keep only the columns we need and take a manageable subset
sst2 = sst2[["sentence", "label"]].dropna().sample(600, random_state=42).reset_index(drop=True)

sst2_train = sst2.iloc[:500]
sst2_test  = sst2.iloc[500:]

print(f"Train: {len(sst2_train)} rows, Test: {len(sst2_test)} rows")
sst2_train.head(3)

### Fit

The DataFrame has one text column (`sentence`) and one label column (`label`). AutoGluon recognizes the text column by its content and routes it through a pretrained language model. No feature engineering or tokenization needed on your end.

See Chapter 17 for a deeper explanation of `time_limit` and `presets`.

In [ ]:
predictor_text = MultiModalPredictor(
    label="label",
    problem_type="binary",
    path="ag_sst2_model"
).fit(
    train_data=sst2_train,
    time_limit=120
)

### Evaluate


In [ ]:
results_text = predictor_text.evaluate(sst2_test, metrics=["roc_auc"])
print(results_text)

AutoGluon's `MultiModalPredictor` optimizes for `roc_auc` internally, which is why the training log reports `val_roc_auc` rather than accuracy. A `roc_auc` around 0.75–0.85 after two minutes on 500 training rows is reasonable for a BERT-style model fine-tuned on a small sample. The key takeaway is not the exact number — it is that you got a working text classifier from a raw Hugging Face dataset with about ten lines of code.


---

## Part 2: Images Only — Plant Disease Classification

The dataset is Beans (Makerere AI Lab, 2020), a collection of RGB photographs of bean leaves across three categories: healthy, angular leaf spot, and bean rust. It is hosted on Hugging Face and is a common benchmark for transfer learning on domain-specific image data.

Unlike Fashion-MNIST, Beans images are in color and share visual characteristics with the natural images that most pretrained vision models were trained on. This means the pretrained features transfer well, and you can expect meaningful accuracy even with a small training set and a short time budget.

`MultiModalPredictor` for image tasks expects a DataFrame with a column of **file paths** pointing to image files on disk. The cell below loads a subset, saves each image as a JPEG, and builds that DataFrame.


In [ ]:
from PIL import Image

# Load Beans dataset from Hugging Face
beans_ds = load_dataset("beans", split="train").select(range(600))

label_names = ["angular_leaf_spot", "bean_rust", "healthy"]

img_dir = "beans_images"
os.makedirs(img_dir, exist_ok=True)

records = []
for i, item in enumerate(beans_ds):
    path = os.path.join(img_dir, f"img_{i}.jpg")
    item["image"].save(path)
    records.append({"image": path, "label": label_names[item["labels"]]})

beans_df = pd.DataFrame(records)
beans_train = beans_df.iloc[:500]
beans_test  = beans_df.iloc[500:]

print(f"Train: {len(beans_train)}, Test: {len(beans_test)}")
print(f"Label distribution:\n{beans_train['label'].value_counts()}")
beans_train.head(3)

### Fit

Notice the only difference from Part 1 is the DataFrame content. The `MultiModalPredictor` call is identical. AutoGluon detects that the `image` column contains file paths and routes it through a pretrained vision model (a ResNet or ViT variant, depending on the preset).

This section benefits significantly from a GPU. Training on CPU will work but may take 10–15 minutes.

In [ ]:
predictor_image = MultiModalPredictor(
    label="label",
    problem_type="multiclass",
    path="ag_beans_model"
).fit(
    train_data=beans_train,
    time_limit=180
)

### Evaluate and Inspect Predictions


In [ ]:
results_image = predictor_image.evaluate(beans_test, metrics=["accuracy"])
print(results_image)

In [ ]:
import matplotlib.pyplot as plt

# Show a sample of predictions alongside the actual images
sample = beans_test.sample(6, random_state=1).reset_index(drop=True)
preds  = predictor_image.predict(sample)

fig, axes = plt.subplots(1, 6, figsize=(14, 3))
for ax, (_, row), pred in zip(axes, sample.iterrows(), preds):
    img = Image.open(row["image"])
    ax.imshow(img)
    color = "green" if pred == row["label"] else "red"
    ax.set_title(f"True: {row['label']}\nPred: {pred}", fontsize=8, color=color)
    ax.axis("off")
plt.tight_layout()
plt.show()

Green titles are correct predictions, red are misclassifications. With 500 training samples and three minutes of fine-tuning, the model typically reaches around 0.80–0.90 accuracy on this three-class problem. The pretrained vision model transfers well here because bean leaf photographs share enough visual structure with the natural images used in pretraining. Misclassifications tend to cluster around images with unusual lighting or partial leaf coverage, which is the kind of edge case you would investigate further in a real research setting.


---

## Part 3: Text + Tabular — Wine Quality Score Prediction

This section is the closest to a real research scenario. The dataset is a sample of wine reviews from Wine Enthusiast magazine, available on Hugging Face. Each row represents a reviewed wine with a mix of structured fields and free text:

- `description`: the written tasting notes (text column)
- `variety`: the grape variety, such as Pinot Noir or Chardonnay (categorical)
- `country`: the wine's country of origin (categorical)
- `price`: the bottle price in USD (numeric)
- `points`: the quality score from 80 to 100 (the prediction target)

The task is to predict the quality score from the description and structured features. This mirrors a pattern that appears regularly in research: some structured metadata fields, some free-form text, all in the same table. The comparison at the end checks whether the tabular columns add anything beyond what the text alone already captures.


In [ ]:
# Load wine reviews from Hugging Face
wine = load_dataset("huggingface-course/wine-reviews", split="train").to_pandas()
wine = wine[["description", "variety", "country", "price", "points"]].dropna()
wine = wine.sample(600, random_state=42).reset_index(drop=True)

wine_train = wine.iloc[:500]
wine_test  = wine.iloc[500:]

print(f"Train: {len(wine_train)}, Test: {len(wine_test)}")
print(f"Points range: {wine['points'].min()}–{wine['points'].max()}, mean: {wine['points'].mean():.1f}")
wine_train.head(3)

### Fit

The DataFrame has one text column (`description`), two categorical columns (`variety`, `country`), one numeric column (`price`), and a continuous label (`points`). AutoGluon routes each column type to the appropriate model component and fuses the learned representations before making a prediction.

Same call as before, with `problem_type="regression"` since we are predicting a continuous score.


In [ ]:
predictor_combined = MultiModalPredictor(
    label="points",
    problem_type="regression",
    path="ag_wine_model"
).fit(
    train_data=wine_train,
    time_limit=180
)

### Evaluate

We evaluate with root mean squared error (RMSE), which measures how far off the predicted scores are from the actual scores on average. A lower RMSE is better. For context, the score range in this dataset is 80–100, so an RMSE of 2–3 points would be a reasonable result for a small dataset and a short training run.


In [ ]:
results_combined = predictor_combined.evaluate(wine_test, metrics=["rmse"])
print("Text + tabular combined:")
print(results_combined)

### Compare: Does the Tabular Data Actually Help?

One practical research question is whether adding structured features improves over text alone. A quick way to check is to retrain on only the description column and compare RMSE.


In [ ]:
# Text-only version for comparison
wine_train_text = wine_train[["description", "points"]]
wine_test_text  = wine_test[["description", "points"]]

predictor_text_only = MultiModalPredictor(
    label="points",
    problem_type="regression",
    path="ag_wine_textonly_model"
).fit(
    train_data=wine_train_text,
    time_limit=120
)

results_text_only = predictor_text_only.evaluate(wine_test_text, metrics=["rmse"])
print("Text only (description):")
print(results_text_only)
print("\nText + tabular combined:")
print(results_combined)

On a small 500-row dataset, the difference between these two versions is often modest. The description text already encodes a lot of information about quality — reviewers tend to use richer vocabulary for higher-scoring wines. But the tabular columns (variety, country, price) carry independent signal: price in particular correlates with quality in ways the text may not always make explicit. On a larger dataset, the combined model typically edges ahead.

This comparison pattern — train once with all features, once without a subset, and compare — is a practical way to check whether each data type is actually contributing.


---

## Summary

All three sections used the same `MultiModalPredictor` interface with different DataFrame structures. The only things that changed were the data and the `problem_type` parameter.

| Section | Columns fed in | AutoGluon routed to |
|---|---|---|
| Text only | 1 text column | Pretrained language model (BERT-family) |
| Images only | 1 image-path column | Pretrained vision model (ResNet / ViT) |
| Text + tabular | 1 text + 2 categorical + 1 numeric | Language model + tabular encoder, fused |

From here, the natural next step is Chapter 21, which covers how to validate and interpret these results in a research context — including when a good accuracy number is not the same as a meaningful research finding.
